# WarehouseSort — H100 run (state Diffusion Policy)

Run every cell top to bottom. Targets the **medium** level (set `DIFFICULTY` below; change to
`"hard"` later and re-run from **Cell 4** onward to reuse everything else as-is).

**The fix this run relies on** (see `NOTES_FOR_TEAMS.md` and `warehouse_sort/il_policy.py`):
the previous rejected submission used a scripted controller fed by a perception network --
disqualified, since it isn't a learned observation→action policy. Separately, the *state*
Diffusion Policy baseline itself had a deployment bug: it resampled the diffusion model from
fresh noise on every single step instead of committing to an action chunk like training's
evaluator does, so the gripper "close" command flickered and grasps never latched. That's fixed
in `il_policy.py` (closed-loop `act_horizon` chunking) -- this is what actually lets a *learned*
policy finish the task instead of leaving boxes behind.

**Expected timing on an H100** (medium, defaults below): installs ~3-6 min (first run only,
cached after), demo download ~1-3 min (first run only), training ~12-18 min, final eval + video
~2-4 min. Total well inside the 20-30 min budget on a warm cache; allow extra on the very first
run for installs/download.

In [ ]:
import os, time, subprocess, sys

T0 = time.time()
assert os.path.exists("pixi.toml"), "Run this notebook from the repo root (where pixi.toml lives)."
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

## 1. Install (idempotent — safe to re-run; cached after the first time)

In [ ]:
import os, subprocess

pixi_bin = os.path.expanduser("~/.pixi/bin")
os.environ["PATH"] = pixi_bin + os.pathsep + os.environ["PATH"]

if subprocess.run(["which", "pixi"], capture_output=True).returncode != 0:
    subprocess.run("curl -fsSL https://pixi.sh/install.sh | bash", shell=True, check=True)

subprocess.run(["pixi", "--version"], check=True)

In [ ]:
subprocess.run(["pixi", "install"], check=True)
# NOTE: no separate "pixi run install" step -- pixi.toml's [pypi-dependencies] already
# declares warehouse_sort as an editable install, so `pixi install` alone covers it.
# (The README's `pixi run install` is stale: this repo defines no such pixi task --
# running it falls through to the unrelated system `install` binary and errors.)

## 2. Get the demonstrations (idempotent — skips re-download if already staged)

Needs either: running on Kaggle (auto-mounted), or a Kaggle API token on this machine
(`~/.kaggle/kaggle.json`, or set `KAGGLE_USERNAME` / `KAGGLE_KEY` below before running).

In [ ]:
# If you don't have ~/.kaggle/kaggle.json on this machine, uncomment and fill these in:
# os.environ["KAGGLE_USERNAME"] = "your-kaggle-username"
# os.environ["KAGGLE_KEY"] = "your-kaggle-api-key"

import glob

if glob.glob("il/demos/*/trajectory.state.pd_ee_delta_pos.physx_cuda.h5"):
    print("demos already staged, skipping download:")
    for f in glob.glob("il/demos/*/trajectory.state.pd_ee_delta_pos.physx_cuda.h5"):
        print(" ", f)
else:
    subprocess.run(["pixi", "run", "python", "il/download_demos.py"], check=True)

## 3. Configure the run

Change only `DIFFICULTY` to reuse this notebook for `hard` later — everything below derives from
it. `MAX_EPISODE_STEPS` follows the ~100×parcels rule from `NOTES_FOR_TEAMS.md` (medium/hard
demos run long enough that the default 200-step cap truncates episodes before every parcel is
placed). `PRED_HORIZON` / `ACT_HORIZON` / `OBS_HORIZON` must match the same values already set as
defaults in `warehouse_sort/il_policy.py:load_dp` — if you change them here, update those
defaults too, or the saved checkpoint won't load correctly at eval time.

In [ ]:
DIFFICULTY = "medium"   # "medium" now; switch to "hard" later and re-run from here

NUM_PARCELS = {"easy": 2, "medium": 4, "hard": 6}[DIFFICULTY]
MAX_EPISODE_STEPS = 100 * NUM_PARCELS                 # medium=400, hard=600
TOTAL_ITERS = {"medium": 30000, "hard": 35000}.get(DIFFICULTY, 20000)
EVAL_FREQ = max(1000, TOTAL_ITERS // 6)

OBS_HORIZON = 2
ACT_HORIZON = 8
PRED_HORIZON = 24       # must match il_policy.py:load_dp's pred_horizon default

EXP_NAME = f"warehouse_state_dp_{DIFFICULTY}"
CKPT_PATH = (f"il/baselines/diffusion_policy/runs/{EXP_NAME}/"
             f"checkpoints/best_eval_sort_accuracy.pt")

print(f"difficulty={DIFFICULTY}  parcels={NUM_PARCELS}  max_episode_steps={MAX_EPISODE_STEPS}")
print(f"total_iters={TOTAL_ITERS}  eval_freq={EVAL_FREQ}")
print(f"horizons: obs={OBS_HORIZON} act={ACT_HORIZON} pred={PRED_HORIZON}")
print(f"checkpoint -> {CKPT_PATH}")

## 4. Train (one checkpoint for this level — state obs size depends on parcel count)

In [ ]:
train_cmd = [
    "pixi", "run", "python", "il/train.py", "method=dp",
    f"demo_dir={DIFFICULTY}",
    f"max_episode_steps={MAX_EPISODE_STEPS}",
    f"flags.total_iters={TOTAL_ITERS}",
    f"flags.eval_freq={EVAL_FREQ}",
    f"flags.obs_horizon={OBS_HORIZON}",
    f"flags.act_horizon={ACT_HORIZON}",
    f"flags.pred_horizon={PRED_HORIZON}",
    f"flags.exp_name={EXP_NAME}",
]
print("running:", " ".join(train_cmd))
t_train0 = time.time()
subprocess.run(train_cmd, check=True)
print(f"\n[train] took {(time.time() - t_train0) / 60:.1f} min")

assert os.path.exists(CKPT_PATH), f"checkpoint not found at {CKPT_PATH} -- check training logs above"
print("checkpoint saved:", CKPT_PATH)

## 5. Evaluate (same interface as judging) — prints sort accuracy + saves a video

In [ ]:
import re

eval_cmd = [
    "pixi", "run", "python", "eval.py",
    f"difficulty={DIFFICULTY}", "obs_mode=state",
    "policy=warehouse_sort.il_policy:load_dp",
    f"checkpoint={CKPT_PATH}",
    "eval_config=conf/eval/default.yaml",
    f"max_episode_steps={MAX_EPISODE_STEPS}",
]
print("running:", " ".join(eval_cmd))
t_eval0 = time.time()
proc = subprocess.run(eval_cmd, capture_output=True, text=True)
if proc.returncode != 0:
    print(proc.stdout[-4000:])
    print(proc.stderr[-4000:])
    raise RuntimeError(f"eval.py failed (exit {proc.returncode}) -- see output above")
print(proc.stdout[-4000:])
print(f"\n[eval] took {(time.time() - t_eval0) / 60:.1f} min")

m = re.search(r"SORT ACCURACY:\s*([\d.]+)\s*%", proc.stdout)
sort_acc = float(m.group(1)) if m else None
vid_dir_match = re.search(r"saved rollout video.*?-> (\S+)", proc.stdout)
vid_dir = vid_dir_match.group(1) if vid_dir_match else None

print("=" * 50)
if sort_acc is None:
    print("Could not parse sort accuracy -- check the full output above.")
elif sort_acc >= 99.99:
    print(f"PASS: sort accuracy = {sort_acc:.1f}%  (difficulty={DIFFICULTY})")
else:
    print(f"sort accuracy = {sort_acc:.1f}%  (difficulty={DIFFICULTY}) -- below 100%.")
    print("See NOTES_FOR_TEAMS.md sec.1-3: check mis_sort_rate/all_placed_rate above to tell\n"
          "a grasp-latch issue (raise act_horizon / num_inference_steps in load_dp, re-eval,\n"
          "no retrain needed) apart from a true capability gap (train longer / more demos).")
print("=" * 50)

## 6. Save the evaluation video to `outputs/<difficulty>_eval.mp4`

In [ ]:
import glob, shutil
from IPython.display import Video, display

if vid_dir and os.path.isdir(vid_dir):
    candidates = sorted(glob.glob(os.path.join(vid_dir, "*.mp4")))
else:
    # fall back to the most recently modified hydra run if the path wasn't parsed
    run_dirs = sorted(glob.glob("outputs/*/*"), key=os.path.getmtime)
    candidates = sorted(glob.glob(os.path.join(run_dirs[-1], "videos", "*.mp4"))) if run_dirs else []

assert candidates, "no eval video found -- check the eval.py output above"
src_video = candidates[0]

out_video = f"outputs/{DIFFICULTY}_eval.mp4"
os.makedirs("outputs", exist_ok=True)
shutil.copy(src_video, out_video)
print(f"video saved -> {out_video}")
display(Video(out_video, embed=True, width=480))

print(f"\nTOTAL wall-clock time: {(time.time() - T0) / 60:.1f} min")

## Next steps

- If `medium` looks good: set `DIFFICULTY = "hard"` in Cell 3 and re-run from there (the demos
  for hard download automatically in Cell 2's check on the next run if not already staged).
- `submission.yaml` at the repo root already points at this medium checkpoint path. Commit the
  checkpoint file at `CKPT_PATH` (it's small — just the U-Net weights) and push:
  ```bash
  git add warehouse_sort/il_policy.py conf/eval/default.yaml submission.yaml run_h100.ipynb \
          il/baselines/diffusion_policy/runs/warehouse_state_dp_medium/checkpoints/best_eval_sort_accuracy.pt
  git commit -m "Add medium state-DP checkpoint"
  git push
  ```